In [102]:
import scanpy as sc
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import anndata
from scipy.stats import median_abs_deviation
import scrublet as scr
import gseapy

In [ ]:
adata = sc.read_10x_h5(filename='/cellranger_GS54/gs54-aggr/outs/count/filtered_feature_bc_matrix.h5')
adata

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 6429 × 36601
    var: 'gene_ids', 'feature_types', 'genome'

In [105]:
sample = []
samples = ['SoftMi', 'SoftNon', 'StiffMi', 'StiffNon']

for umi in adata.obs_names:
    sample.append(samples[int(umi[-1])-1])

sample

adata.obs['sample'] = sample

In [106]:
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 6429 × 36601
    obs: 'sample'
    var: 'gene_ids', 'feature_types', 'genome'

# Quality Control

## Filter low quality reads

In [107]:
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))

In [108]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)
adata

AnnData object with n_obs × n_vars = 6429 × 36601
    obs: 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'

In [109]:

p2 = sc.pl.violin(adata, "pct_counts_mt", groupby='sample')
#p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(


In [110]:
sc.pl.violin(adata, "total_counts", groupby='sample', log=True, cut=0)
plt.show()

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(


In [112]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

In [113]:
adata.obs["outlier"] = (
    is_outlier(adata, "log1p_total_counts", 5)
    | is_outlier(adata, "log1p_n_genes_by_counts", 5)
    | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
)
adata.obs.outlier.value_counts()

outlier
False    5948
True      481
Name: count, dtype: int64

### Filter out low gene and cell counts

In [114]:
mad_cutoff=3

In [115]:
print('Number of genes before filtering: {:d}'.format(adata.n_vars))
print('Number of cells before filtering: {:d}'.format(adata.n_obs))

print('Removing', np.sum(is_outlier(adata, "log1p_total_counts", mad_cutoff) | is_outlier(adata, "log1p_n_genes_by_counts", mad_cutoff)), 'cells that don\'t meet MAD')
adata = adata[~(is_outlier(adata, "log1p_total_counts", mad_cutoff) | is_outlier(adata, "log1p_n_genes_by_counts", mad_cutoff))]

print("Removing", sum(adata.obs['pct_counts_mt'] > 20), "cells with mt-frac > 20%")
adata = adata[adata.obs['pct_counts_mt'] < 20]

#min 20 cells- filters out 0 count genes
sc.pp.filter_genes(adata, min_cells=20)

print('Number of genes after filtering: {:d}'.format(adata.n_vars))
print('Number of cells after filtering: {:d}'.format(adata.n_obs))

Number of genes before filtering: 36601
Number of cells before filtering: 6429
Removing 1486 cells that don't meet MAD
Removing 0 cells with mt-frac > 20%
Number of genes after filtering: 9619
Number of cells after filtering: 4943


/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:250: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var['n_cells'] = number


In [116]:
plt.hist(adata.obs['total_counts'], bins=60)
plt.title('number of counts')

plt.figure()
plt.hist(adata.obs['n_genes_by_counts'], bins=60)
plt.title('number of genes')
plt.show()

### Filter out cells with high mitrochrondrial dna (>20% mt)

In [117]:
p2 = sc.pl.violin(adata, "pct_counts_mt", groupby='sample')
#p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
sc.pl.violin(adata, "total_counts", groupby='sample', log=True, cut=0)
plt.show()

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_utils.py:626: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  pl.figure(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/scanpy/plotting/_anndata.py:839: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(
/

### Doublet Detection

In [119]:
# Given a raw (unnormalized) UMI counts matrix counts_matrix with cells as rows and genes as columns, calculate a doublet score for each cell:

scrub = scr.Scrublet(adata.X, expected_doublet_rate=0.023)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.34
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 1.9%
Overall doublet rate:
	Expected   = 2.3%
	Estimated  = 8.4%
Elapsed time: 5.0 seconds


In [120]:
scrub.plot_histogram()

(<Figure size 800x300 with 2 Axes>,
 array([<Axes: title={'center': 'Observed transcriptomes'}, xlabel='Doublet score', ylabel='Prob. density'>,
        <Axes: title={'center': 'Simulated doublets'}, xlabel='Doublet score', ylabel='Prob. density'>],
       dtype=object))

In [121]:
#2d visualization of doublets

scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))

scrub.plot_embedding('UMAP', order_points=True);

/stor/home/tms4478/miniconda3/envs/scrna/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [122]:
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublets"] = predicted_doublets
adata.obs.predicted_doublets.value_counts()

predicted_doublets
False    4935
True        8
Name: count, dtype: int64

### Correction of ambient RNA

## Normalize 

In [128]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata)
adata.layers['cpm'] = adata.X.copy()
sc.pp.log1p(adata)
adata.layers['norm_counts'] = adata.X.copy()

In [ ]:
geneFileLoc = 'Macosko_cell_cycle_genes.txt'
cc_genes = pd.read_table(geneFileLoc, delimiter='\t')
s_genes = cc_genes['S'].dropna()
g2m_genes = cc_genes['G2.M'].dropna()

s_genes_mm = [gene.lower() for gene in s_genes]
g2m_genes_mm = [gene.lower() for gene in g2m_genes]

def cc_regress(adata):
    s_genes_mm_ens = adata.var_names[np.in1d([i.lower() for i in adata.var_names], s_genes_mm)]
    g2m_genes_mm_ens = adata.var_names[np.in1d([i.lower() for i in adata.var_names], g2m_genes_mm)]

    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes_mm_ens, g2m_genes=g2m_genes_mm_ens)
    adata.obs['cc_difference'] = adata.obs['S_score'] - adata.obs['G2M_score']
    sc.pp.regress_out(adata, 'cc_difference')
    adata.uns['Processing'] = ['QC','Normalized','Cell Cycle Regressed']

    return adata

In [137]:
adata = cc_regress(adata)

In [140]:
adata.layers['cc_regressed'] = adata.X.copy()
adata.X = adata.layers['norm_counts'].copy()

In [141]:
anndata.AnnData.write(adata, 'GS54_cleaned.h5ad')